In [5]:
!pip install -q tensorflow-datasets

In [14]:
import urllib.request

url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

urllib.request.urlretrieve(url, "catsdogs.zip")

print("Download complete!")

ConnectionAbortedError: [WinError 10053] An established connection was aborted by the software in your host machine

In [13]:
!unzip -q catsdogs.zip

[catsdogs.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in catsdogs.zip,
        and cannot find catsdogs.zip.zip, period.


In [10]:
import os
print(os.listdir("PetImages"))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'PetImages'

In [ ]:
from pathlib import Path
from PIL import Image

for folder in ["PetImages/Cat", "PetImages/Dog"]:
    for img_path in Path(folder).glob("*.jpg"):
        try:
            Image.open(img_path).verify()
        except:
            img_path.unlink()

In [ ]:
from torchvision import transforms
import torch
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [ ]:
full_dataset = datasets.ImageFolder(
    root="PetImages",
    transform=train_transform
)

print(full_dataset.classes)
print(full_dataset.class_to_idx)
print(len(full_dataset))

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_data, test_data = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply test transform
test_data.dataset.transform = test_transform

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=256,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_data,
    batch_size=256,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)

In [ ]:
import torch
from torch import nn

class Classification(nn.Module):
    def __init__(self, input_shape=3, hidden_units=64):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(input_shape, hidden_units, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(inplace=True),

            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(hidden_units, hidden_units * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(inplace=True),

            nn.Conv2d(hidden_units * 2, hidden_units * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(hidden_units * 2, hidden_units * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 4),
            nn.ReLU(inplace=True),

            nn.Conv2d(hidden_units * 4, hidden_units * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 4),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 4 (NEW)
            nn.Conv2d(hidden_units * 4, hidden_units * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 8),
            nn.ReLU(inplace=True),

            nn.Conv2d(hidden_units * 8, hidden_units * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 8),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),

            nn.Linear(hidden_units * 8, 512),
            nn.ReLU(inplace=True),

            nn.Dropout(0.5),

            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)

model_2 = Classification(
    input_shape=3,
    hidden_units=64
).to(device)

print(model_2)

In [ ]:

import torch.optim as optim

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model_2.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [15]:
from tqdm.auto import tqdmfba

# Number of epochs
epochs = 50

# Keep track of the best accuracy
best_acc = 0.0

for epoch in range(epochs):

    print(f"Epoch: {epoch+1}/{epochs}")
    print("-" * 30)

    # =======================
    # Training
    # =======================
    model_2.train()

    train_loss = 0
    train_correct = 0
    train_total = 0fbatch

    for X, y in tqdm(train_loader):

        X = X.to(device)
        y = y.float().unsqueeze(1).to(device)

        # Forward pass
        y_logits = model_2(X)

        # Loss
        loss = loss_fn(y_logits, y)

        train_loss += loss.item()

        # Predictions
        probs = torch.sigmoid(y_logits)
        y_pred = (probs >= 0.5).float()

        train_correct += (y_pred == y).sum().item()
        train_total += y.size(0)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_loader)
    train_acc = (train_correct / train_total) * 100

    # =======================
    # Testing
    # =======================
    model_2.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.inference_mode():

        for X, y in test_loader:

            X = X.to(device)
            y = y.float().unsqueeze(1).to(device)

            # Forward pass
            test_logits = model_2(X)

            # Loss
            loss = loss_fn(test_logits, y)

            test_loss += loss.item()

            # Predictions
            probs = torch.sigmoid(test_logits)
            test_pred = (probs >= 0.5).float()

            test_correct += (test_pred == y).sum().item()
            test_total += y.size(0)

    test_loss /= len(test_loader)
    test_acc = (test_correct / test_total) * 100

    # =======================
    # Results
    # =======================
    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}%"
    )

    # =======================
    # Save Best Model
    # =======================
    if test_acc > best_acc:
        best_acc = test_acc

        torch.save({
            'epoch': epoch,
            'model_state_dict': model_2.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc
        }, "best_model.pth")

        print(f"✅ Best model saved! Test Accuracy: {best_acc:.2f}%")

print("\nTraining completed!")
print(f"Best Test Accuracy: {best_acc:.2f}%")

Epoch: 1/50
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.6265 | Train Acc: 63.78% | Test Loss: 0.6523 | Test Acc: 65.52%
✅ Best model saved! Test Accuracy: 65.52%
Epoch: 2/50
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.5591 | Train Acc: 71.18% | Test Loss: 0.9276 | Test Acc: 62.48%
Epoch: 3/50
------------------------------


...

Train Loss: 0.0088 | Train Acc: 99.70% | Test Loss: 0.4414 | Test Acc: 91.68%
Epoch: 49/50
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0151 | Train Acc: 99.47% | Test Loss: 0.2108 | Test Acc: 94.60%
✅ Best model saved! Test Accuracy: 94.60%
Epoch: 50/50
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0111 | Train Acc: 99.58% | Test Loss: 0.2202 | Test Acc: 94.66%
✅ Best model saved! Test Accuracy: 94.66%

Training completed!
Best Test Accuracy: 94.66%


In [16]:
checkpoint = torch.load("best_model.pth")

model_2.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1

In [17]:
epochs = start_epoch + 40

In [18]:
epochs

90

In [19]:
from tqdm.auto import tqdm
import torch

# ==========================
# LOAD CHECKPOINT
# ==========================
checkpoint = torch.load("best_model.pth")

model_2.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_acc = checkpoint["best_acc"]

print(f"Resuming from epoch {start_epoch}")
print(f"Current best accuracy: {best_acc:.2f}%")

# Train 40 more epochs
epochs = start_epoch + 40

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(start_epoch, epochs):

    print(f"\nEpoch: {epoch+1}/{epochs}")
    print("-" * 30)

    # =======================
    # TRAINING
    # =======================
    model_2.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for X, y in tqdm(train_loader):

        X = X.to(device)
        y = y.float().unsqueeze(1).to(device)

        y_logits = model_2(X)

        loss = loss_fn(y_logits, y)

        train_loss += loss.item()

        probs = torch.sigmoid(y_logits)
        y_pred = (probs >= 0.5).float()

        train_correct += (y_pred == y).sum().item()
        train_total += y.size(0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total

    # =======================
    # VALIDATION
    # =======================
    model_2.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.inference_mode():

        for X, y in test_loader:

            X = X.to(device)
            y = y.float().unsqueeze(1).to(device)

            test_logits = model_2(X)

            loss = loss_fn(test_logits, y)

            test_loss += loss.item()

            probs = torch.sigmoid(test_logits)
            test_pred = (probs >= 0.5).float()

            test_correct += (test_pred == y).sum().item()
            test_total += y.size(0)

    test_loss /= len(test_loader)
    test_acc = 100 * test_correct / test_total

    # =======================
    # RESULTS
    # =======================
    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}%"
    )

    # =======================
    # SAVE BEST MODEL
    # =======================
    if test_acc > best_acc:

        best_acc = test_acc

        torch.save({
            'epoch': epoch,
            'model_state_dict': model_2.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc
        }, "best_model.pth")

        print(f"✅ New Best Model Saved! Accuracy = {best_acc:.2f}%")

    # =======================
    # SAVE CHECKPOINT
    # =======================
    torch.save({
        'epoch': epoch,
        'model_state_dict': model_2.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_acc': best_acc
    }, "checkpoint.pth")

print(f"\nTraining Complete!")
print(f"Best Accuracy Achieved: {best_acc:.2f}%")

Resuming from epoch 50
Current best accuracy: 94.66%

Epoch: 51/90
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0149 | Train Acc: 99.42% | Test Loss: 0.8416 | Test Acc: 87.42%

Epoch: 52/90
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0210 | Train Acc: 99.22% | Test Loss: 0.2256 | Test Acc: 93.92%

Epoch: 53/90
------------------------------


...

Train Loss: 0.0050 | Train Acc: 99.82% | Test Loss: 0.2109 | Test Acc: 95.58%

Epoch: 89/90
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0026 | Train Acc: 99.91% | Test Loss: 0.3518 | Test Acc: 93.36%

Epoch: 90/90
------------------------------


  0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0030 | Train Acc: 99.89% | Test Loss: 0.2743 | Test Acc: 94.90%

Training Complete!
Best Accuracy Achieved: 95.62%


In [20]:
torch.save({
    'model_state_dict': model_2.state_dict(),
    'best_acc': 95.58
}, "best_model.pth_final")

In [21]:
!ls

 CDLA-Permissive-2.0.pdf   best_model.pth_final   checkpoint.pth
 PetImages		   catdog.ipynb		  main.py
 best_model.pth		   catsdogs.zip		 'readme[1].txt'


In [22]:
!python main.py

  File "/teamspace/studios/this_studio/main.py", line 1
    !pip install fastapi uvicorn pillow torch torchvision
    ^
SyntaxError: invalid syntax


In [4]:
import torch

model = Classification().to(device);

checkpoint = torch.load("best_model.pth", map_location=device);
model.load_state_dict(checkpoint["model_state_dict"]);

model.eval()

# Evaluate
correct = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device).float()

        logits = model(images).squeeze()

        probs = torch.sigmoid(logits)

        preds = (probs >= 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print("=" * 50)
print(f"Test Accuracy: {accuracy:.2f}%")
print("=" * 50)

# Class mapping
print("Classes:", full_dataset.classes)
print("Class Mapping:", full_dataset.class_to_idx)

# Weight sanity check
weight_sum = 0

for p in model.parameters():
    weight_sum += p.abs().sum().item()

print("Weight Sum:", weight_sum)

'.' expected
import torch
            ^
';' expected
model = Classification().to(device);
     ^
cannot find symbol
  symbol:   variable checkpoint
checkpoint = torch.load("best_model.pth", map_location=device);
^--------^
cannot find symbol
  symbol:   variable map_location
checkpoint = torch.load("best_model.pth", map_location=device);
                                          ^----------^
cannot find symbol
  symbol:   variable device
checkpoint = torch.load("best_model.pth", map_location=device);
                                                       ^----^
cannot find symbol
  symbol:   variable torch
checkpoint = torch.load("best_model.pth", map_location=device);
             ^---^
cannot find symbol
  symbol:   variable checkpoint
model.load_state_dict(checkpoint["model_state_dict"]);
                      ^--------^
incompatible types: java.lang.String cannot be converted to int
model.load_state_dict(checkpoint["model_state_dict"]);
                                 ^-----------

In [9]:
import torch
from torch import nn

test_correct = 0
test_total = 0

model.eval()

with torch.inference_mode():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        labels = labels.unsqueeze(1).float()

        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

test_acc = 100 * test_correct / test_total

print(f"Test Accuracy: {test_acc:.2f}%")

NameError: name 'model' is not defined

In [ ]:
model.load_state_dict(
    torch.load("best_model.pth", map_location=device)
)